# Weights & Biases: your first 15 minutes

## Goal
Learn the core application programming interface (API) by running short cells from top to bottom.
No dataset, model, or training is required. All metric values below are invented for this demo.

**Workflow:** sign in → start a run → log values → save a file → finish the run

| Term | Meaning in this demo |
|---|---|
| Project | A folder for related runs |
| Run | One recorded execution |
| Config | Settings recorded for that execution |
| Metric | A number recorded as the run progresses |
| Summary | A final value or note for the run |
| Artifact | A versioned bundle of files |

**Timing:** setup 3 min; start a run 3 min; log values 4 min; save and finish 3 min; inspect 2 min

## Setup
### 1. Install the package
`%pip` installs into this notebook's Python environment. Run this once.
If Jupyter requests a kernel restart, restart it before continuing.

In [ ]:
%pip install -q wandb

### 2. Sign in
Create or sign in to a [Weights & Biases account](https://wandb.ai), then run this cell.
`wandb.login()` connects Python to your account. Follow its prompt; if it requests an API key,
use [the authorization page](https://wandb.ai/authorize). Enter the key in the prompt, never in a code cell.
If already authenticated, it can reuse your saved login. Login does not create a run.

[Official getting-started example](https://github.com/wandb/wandb#quickstart)

In [ ]:
import wandb

wandb.login()

## Steps
### 3. Choose a project
These are ordinary Python variables. Nothing has been sent to W&B yet.
`ENTITY` selects the account or team workspace. Leave it as `None` to use your default,
or replace it with a workspace slug you can write to, such as `"your-team-name"`.

In [ ]:
PROJECT = "wandb-intro"
ENTITY = None

### 4. Start a run and record its settings
`wandb.init()` starts a run and creates the project if needed and permitted.
`name` is a readable label; W&B assigns a separate run identifier.
`config` records settings, but does not execute any training.
Open the printed link to watch the run update.

[Official quickstart](https://github.com/wandb/wandb#quickstart)

In [ ]:
run = wandb.init(
    project=PROJECT,
    entity=ENTITY,
    name="my-first-run",
    config={"learning_rate": 0.01, "purpose": "API demo"},
)

print("Run link:", run.url)
print("Recorded learning rate:", run.config["learning_rate"])

### 5. Log one number
`run.log()` records a dictionary of metric names and values.
We choose the name `demo_score`; it has no scientific meaning here.
W&B uses an automatic step counter, starting at 0, when no step is supplied.

In [ ]:
run.log({"demo_score": 0.50})

### 6. Log progress
Each call adds another point to the metric history. This produces four points in total,
including the previous cell. Open the run's workspace to see the chart.
This loop sends invented numbers; it does not train a model.

In [ ]:
for score in [0.65, 0.75, 0.85]:
    run.log({"demo_score": score})

### 7. Record a final summary
Use `run.summary` for a result or note you want to find without reading the whole history.
Here we explicitly record the largest value we entered. W&B does not calculate this maximum for us.

In [ ]:
run.summary["best_demo_score"] = 0.85
run.summary["note"] = "Intro complete; invented values only"

### 8. Create a tiny file
This is standard Python. It creates a local file, with no upload yet.

In [ ]:
from pathlib import Path

note_path = Path("wandb_demo_note.txt")
note_path.write_text("Hello from my first W&B run!\n", encoding="utf-8")
print(note_path.read_text(encoding="utf-8"))

### 9. Save the file as an artifact
`wandb.Artifact()` creates a bundle; `add_file()` puts the file in it;
`run.log_artifact()` uploads it and records it as an output of this run.
`note` is a label we choose for the artifact type.
This saves a versioned file to the project. Linking an artifact to W&B Registry is a separate operation.

[Official artifact guide](https://docs.coreweave.com/products/wandb/artifacts/construct-an-artifact)

In [ ]:
artifact = wandb.Artifact(name="demo-note", type="note")
artifact.add_file(str(note_path))
run.log_artifact(artifact)

### 10. Finish the run
`run.finish()` waits for pending uploads and closes this run.
Your logged values and artifact remain available in W&B.
To record another experiment, start another run with `wandb.init()`.

[Official run lifecycle example](https://github.com/wandb/wandb#quickstart)

In [ ]:
run.finish()
print("Finished. Open the run link from step 4.")

## Next steps
### Optional: reuse the file in a second run, 2 extra minutes
Run this only after step 10 has completed successfully online.
`use_artifact()` records the artifact as this second run's input;
`download()` retrieves its files. `latest` selects the latest version in this artifact collection.
The first run produces a file, and the second consumes it. W&B records that dependency.

`with wandb.init(...)` finishes the run automatically when the block ends;
an unhandled exception marks the run as failed. You do not need a separate `finish()` in this pattern.

[Official artifact reuse guide](https://docs.coreweave.com/products/wandb/artifacts/download-and-use-an-artifact)

In [ ]:
with wandb.init(project=PROJECT, entity=run.entity, name="read-note") as reader:
    saved_note = reader.use_artifact("demo-note:latest")
    folder = saved_note.download()
    text = (Path(folder) / "wandb_demo_note.txt").read_text(encoding="utf-8")
    print(text)
    reader.summary["file_read"] = True

### Command reference

| Command | Purpose |
|---|---|
| `wandb.login()` | Authenticate |
| `wandb.init(...)` | Start a run |
| `run.config["setting"]` | Read a recorded setting |
| `run.log({...})` | Record metric history |
| `run.summary["result"] = value` | Record a final value or note |
| `wandb.Artifact(...)` | Create a file bundle |
| `artifact.add_file(...)` | Add a local file to the bundle |
| `run.log_artifact(artifact)` | Save the bundle as a run output |
| `run.finish()` | Flush pending uploads and close the run |
| `run.use_artifact("name:latest")` | Record an artifact as a run input |
| `artifact.download()` | Download the artifact files |

**Remember:** config records settings; log records progress; summary records final results; artifacts record files.